<a href="https://colab.research.google.com/github/No1Hoo/PodForge/blob/master/colab/voxcpm_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PodForge — VoxCPM2 TTS Server

This notebook starts a VoxCPM2 server on Google Colab's free T4 GPU.

**Steps:**
1. Run all cells (Runtime → Run all)
2. Copy the ngrok public URL printed below
3. Set it as `TTS_BASE_URL` in PodForge backend

Requirements: Colab with T4 GPU (Runtime → Change runtime type → T4 GPU)

In [1]:
# Install dependencies
!pip install -q voxcpm fastapi uvicorn pyngrok soundfile numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 19.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 31.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Configure ngrok (required for public URL tunneling)
from pyngrok import ngrok
ngrok.set_auth_token("3EOr384hN9MABHuFYtoCMY6Ujb5_6UDb37kBKe1Mx6E5DHUr8")
print("ngrok configured ✓")

ngrok configured ✓


In [3]:
# Download and load model (~4GB, first run only)
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print(f"Model loaded! Sample rate: {model.tts_model.sample_rate}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

voxcpm_model_path: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d, zipenhancer_model_path: None, enable_denoiser: False
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensors: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/model.safetensors
Loaded VoxCPM2Model
Warm up VoxCPMModel...
  0%|          | 0/10 [00:00<?, ?it/s]W0531 04:32:48.093000 1146 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/c

Model loaded! Sample rate: 48000


In [ ]:
# Start TTS server with ngrok tunnel
import io
import base64
import threading
import numpy as np
import soundfile as sf
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

# Auto-load model if not already loaded
try:
    model
    print("Model already loaded ✓")
except NameError:
    print("Loading model...")
    from voxcpm import VoxCPM
    model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
    print(f"Model loaded! Sample rate: {model.tts_model.sample_rate}")

app = FastAPI(title="PodForge TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    reference_wav_path: str | None = None

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: TTSRequest):
    kwargs = {
        "text": req.text,
        "cfg_value": req.cfg_value,
        "inference_timesteps": req.inference_timesteps,
    }
    if req.reference_wav_path:
        kwargs["reference_wav_path"] = req.reference_wav_path

    wav = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, wav, model.tts_model.sample_rate, format="WAV")
    buf.seek(0)
    audio_b64 = base64.b64encode(buf.read()).decode()

    return {
        "audio_base64": audio_b64,
        "sample_rate": model.tts_model.sample_rate,
        "duration_seconds": len(wav) / model.tts_model.sample_rate,
    }

# Kill any existing process on port 8809
import subprocess
subprocess.run(["fuser", "-k", "8809/tcp"], capture_output=True)
import time; time.sleep(1)

# Start ngrok tunnel
public_url = ngrok.connect(8809)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"Copy this URL and set it as TTS_BASE_URL in PodForge!")
print(f"{'='*60}\n")

# Run server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8809)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("TTS server is running! Keep this cell running.")
print("Press Ctrl+C or stop the cell to shut down.")

# Keep alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()

Model already loaded ✓


INFO:     Started server process [1146]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8809 (Press CTRL+C to quit)



PUBLIC URL: NgrokTunnel: "https://subsystem-twilight-handwork.ngrok-free.dev" -> "http://localhost:8809"
Copy this URL and set it as TTS_BASE_URL in PodForge!

TTS server is running! Keep this cell running.
Press Ctrl+C or stop the cell to shut down.
INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2401:d560:0:107:1c00:5fff:fe00:53a:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "GET /health HTTP/1.1" 200 OK


 12%|█▎        | 59/472 [00:21<02:28,  2.78it/s]

INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "POST /generate HTTP/1.1" 200 OK


...

INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "POST /generate HTTP/1.1" 200 OK



 11%|█         | 45/412 [00:16<02:13,  2.76it/s]

INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "POST /generate HTTP/1.1" 200 OK



 12%|█▏        | 34/274 [00:12<01:27,  2.74it/s]

INFO:     2401:d560:0:107:1c00:64ff:fe00:591:0 - "POST /generate HTTP/1.1" 200 OK


In [ ]:
# Start TTS server with ngrok tunnel
import io
import base64
import threading
import numpy as np
import soundfile as sf
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

app = FastAPI(title="PodForge TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    reference_wav_path: str | None = None

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: TTSRequest):
    kwargs = {
        "text": req.text,
        "cfg_value": req.cfg_value,
        "inference_timesteps": req.inference_timesteps,
    }
    if req.reference_wav_path:
        kwargs["reference_wav_path"] = req.reference_wav_path

    wav = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, wav, model.tts_model.sample_rate, format="WAV")
    buf.seek(0)
    audio_b64 = base64.b64encode(buf.read()).decode()

    return {
        "audio_base64": audio_b64,
        "sample_rate": model.tts_model.sample_rate,
        "duration_seconds": len(wav) / model.tts_model.sample_rate,
    }

# Start ngrok tunnel
public_url = ngrok.connect(8809)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"Copy this URL and set it as TTS_BASE_URL in PodForge!")
print(f"{'='*60}\n")

# Run server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8809)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("TTS server is running! Keep this cell running.")
print("Press Ctrl+C or stop the cell to shut down.")

# Keep alive
try:
    while True:
        import time
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()